# Data Exploration
- Inspect patient files and recordings, then explore labels and patient metadata.
- No audio cleaning or dataset splitting is performed here.

#### Locating dataset files

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("dark_background")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "raw" / "training_data"
OUTPUT_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

wav_files = sorted(DATA_DIR.rglob("*.wav"))
txt_files = sorted(DATA_DIR.rglob("*.txt"))
tsv_files = sorted(DATA_DIR.rglob("*.tsv"))

print("Audio recordings:", len(wav_files))
print("Text files:", len(txt_files))
print("Annotation files:", len(tsv_files))

if not wav_files:
    raise FileNotFoundError("No WAV files found. Check DATA_DIR.")

#### Patient inspection
- Inspect one patient file directly.
- A patient labeled Present does not necessarily have an audible murmur at every recording location. The index uses the patient label.

In [ ]:
patient_files = [
    path for path in txt_files
    if any(line.startswith("#Murmur:") for line in path.read_text().splitlines())
]

if not patient_files:
    raise FileNotFoundError("No patient text files found.")

example_patient = patient_files[0]
print("Patient file:", example_patient.name)
print(example_patient.read_text())

#### Loading the indexes
- Create missing indexes using the existing script. Otherwise load them without overwriting split assignments.
- After changing raw data, rebuild indexes and rerun the splitting script.

In [ ]:
sys.path.insert(0, str(ROOT))
from src.preprocessing.create_indexes import create_indexes

if not all((OUTPUT_DIR / name).exists() for name in ["patient_index.csv", "recording_index.csv"]):
    create_indexes()

recordings = pd.read_csv(
    OUTPUT_DIR / "recording_index.csv",
    dtype={"patient_id": str}
)
patients = pd.read_csv(
    OUTPUT_DIR / "patient_index.csv",
    dtype={"patient_id": str}
)

#### Checking dataset coverage
- Count patients separately from recordings.
- Check for unindexed audio and patients without indexed recordings.

In [ ]:
indexed_paths = set(recordings["audio_path"])
raw_paths = {path.relative_to(ROOT).as_posix() for path in wav_files}
unindexed_audio = sorted(raw_paths - indexed_paths)
patients_without_audio = patients[
    ~patients["patient_id"].isin(recordings["patient_id"])
]

print("Patients:", patients["patient_id"].nunique())
print("Indexed recordings:", len(recordings))
print("Duplicate patient IDs:", patients["patient_id"].duplicated().sum())
print("Duplicate recording paths:", recordings["audio_path"].duplicated().sum())
print("Recordings without patient metadata:",
      (~recordings["patient_id"].isin(patients["patient_id"])).sum())
print("Patients without indexed audio:", len(patients_without_audio))
print("Unindexed WAV files:", len(unindexed_audio))

if unindexed_audio:
    print((pd.DataFrame({"audio_path": unindexed_audio})).to_string())
if not patients_without_audio.empty:
    print((patients_without_audio).to_string())

print("\nRecordings per patient:")
recording_counts = recordings.groupby("patient_id").size()
print((patients["patient_id"].map(recording_counts).fillna(0).describe()).to_string())


#### Murmur distribution
- Unknown labels are shown here but excluded from the planned Present/Absent training task.
- The majority-class accuracy shows why accuracy alone can be misleading.

In [ ]:
label_order = ["Absent", "Present", "Unknown"]
label_colors = ["steelblue", "salmon", "gold"]
patient_counts = patients["murmur_label"].value_counts().reindex(label_order, fill_value=0)
label_summary = pd.DataFrame({
    "patients": patient_counts,
    "percent": (patient_counts / len(patients) * 100).round(2),
    "recordings": recordings["murmur_label"].value_counts().reindex(label_order, fill_value=0)
})
print((label_summary).to_string())

known_counts = patient_counts.loc[["Absent", "Present"]]
print("Patients with known labels:", known_counts.sum())
if known_counts.sum() > 0:
    print("Majority-class accuracy: {:.2f}%".format(
        known_counts.max() / known_counts.sum() * 100
    ))

patient_counts.plot(kind="bar", color=label_colors, figsize=(8, 4))
plt.title("Patient Murmur Labels")
plt.xlabel("Murmur label")
plt.ylabel("Patients")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "murmur_distribution.png", dpi=300)
plt.show()

#### Recording characteristics
- Locations describe where audio was recorded, not where a murmur was confirmed.
- Durations and sample rates help guide later windowing and resampling decisions.

In [ ]:
print("Sample rates:")
print(recordings["sample_rate"].value_counts())
print("\nChannels:")
print(recordings["channels"].value_counts())
print("\nRecording locations:")
print(recordings["location"].value_counts())
print("\nDuration summary (seconds):")
print(recordings["duration_seconds"].describe())
print("\nNonpositive durations:", (recordings["duration_seconds"] <= 0).sum())

recordings["duration_seconds"].plot(kind="hist", bins=30, figsize=(8, 4))
plt.title("Recording Durations")
plt.xlabel("Seconds")
plt.ylabel("Recordings")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "recording_durations.png", dpi=300)
plt.show()

#### Loading patient metadata
- Locate training_data.csv recursively because the raw files may be in subfolders.
- Use only indexed patients for demographic comparisons. Report any unmatched IDs.

In [ ]:
metadata_files = sorted((ROOT / "data" / "raw").rglob("training_data.csv"))
if len(metadata_files) != 1:
    raise ValueError("Expected one training_data.csv under data/raw. Found: "
                     + str(metadata_files))

metadata = pd.read_csv(metadata_files[0], dtype={"Patient ID": str})
metadata = metadata.rename(columns={"Patient ID": "patient_id"})
if metadata["patient_id"].duplicated().any():
    raise ValueError("Duplicate patient IDs in training_data.csv.")

print("Metadata patients without indexed patient records:",
      (~metadata["patient_id"].isin(patients["patient_id"])).sum())
print("Indexed patients without CSV metadata:",
      (~patients["patient_id"].isin(metadata["patient_id"])).sum())

patient_metadata = patients.merge(
    metadata, on="patient_id", how="left", validate="one_to_one"
)
label_mismatch = patient_metadata["Murmur"].notna() & (
    patient_metadata["murmur_label"] != patient_metadata["Murmur"]
)
print("Conflicting murmur labels:", label_mismatch.sum())
if label_mismatch.any():
    print((patient_metadata.loc[label_mismatch, ["patient_id", "murmur_label", "Murmur"]]).to_string())
    raise ValueError("Resolve conflicting labels before continuing.")

#### Age and sex
- Compare percentages within each group and show group sizes. Missing values remain visible.
- Small groups should be interpreted cautiously.

In [ ]:
for column in ["Age", "Sex"]:
    groups = patient_metadata[column].fillna("Missing")
    counts = pd.crosstab(groups, patient_metadata["murmur_label"])
    counts = counts.reindex(columns=label_order, fill_value=0)
    percentages = counts.div(counts.sum(axis=1), axis=0) * 100

    print(column + " group sizes:")
    print((counts.sum(axis=1).rename("patients").to_frame()).to_string())
    print("Murmur percentages:")
    print((percentages.round(2)).to_string())

    percentages.plot(kind="bar", color=label_colors, figsize=(9, 4))
    plt.title("Murmur Labels by " + column)
    plt.ylabel("Patients within group (%)")
    plt.xticks(rotation=45 if column == "Age" else 0)
    plt.legend(title="Murmur label")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / ("murmur_by_" + column.lower() + ".png"), dpi=300)
    plt.show()

#### Metadata notes
- Murmur-specific fields may be blank when they do not apply.
- These fields are excluded from the audio model inputs.
- Missing ages remain visible in demographic comparisons.

#### Findings and next steps
- Review coverage issues before preparing audio. These checks describe the local download; they do not prove that the full dataset is present.
- Exclude Unknown labels from binary training and report how many patients and recordings are removed.
- Evaluate precision, recall, F1, and the confusion matrix alongside accuracy.
- Split by patient: all recordings and windows from one patient must stay in the same split. Use the existing splitting script during preprocessing.
- Choose filtering, normalization, and window length through audio exploration. Do not trim every recording based on one example.
- Keep patient IDs, file paths, and murmur annotations out of the model input features.